# Geo Taxonomy Ingestion

Ingest `../../resources/Hextile1deg.json` and normalize the `Coordinates` field for polygon use.

----

## Setup

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go

---

## Ingestion

In [3]:
def parse_coordinates(value):
    """Parse polygon coordinates from list/tuple or JSON string."""
    if isinstance(value, str):
        parsed = json.loads(value)
    elif isinstance(value, (list, tuple)):
        parsed = value
    else:
        raise TypeError(f"Unsupported Coordinates type: {type(value)}")

    # Normalize each [lon, lat] pair to float values.
    return [[float(lon), float(lat)] for lon, lat in parsed]

In [4]:
input_path = Path("../../resources/Hextile1deg.json")
with input_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)
df.head()

,Taxonomy,Tag,CenterLon,CenterLat,Coordinates
0,Hextile1deg,tile00001,-124.5,42.435245,"[[-125.0, 42.7239199200323], [-125.0, 42.14656..."
1,Hextile1deg,tile00002,-124.5,47.631397,"[[-125.0, 47.92007234273893], [-125.0, 47.3427..."
2,Hextile1deg,tile00003,-124.0,39.837169,"[[-124.5, 40.125843708678985], [-124.5, 39.548..."
3,Hextile1deg,tile00004,-124.0,41.569219,"[[-124.5, 41.85789451624787], [-124.5, 41.2805..."
4,Hextile1deg,tile00005,-124.0,43.301270,"[[-124.5, 43.58994532381674], [-124.5, 43.0125..."


In [6]:
df["PolygonCoordinates"] = df["Coordinates"].apply(parse_coordinates)
df[["Tag", "PolygonCoordinates"]].head()

,Tag,PolygonCoordinates
0,tile00001,"[[-125.0, 42.7239199200323], [-125.0, 42.14656..."
1,tile00002,"[[-125.0, 47.92007234273893], [-125.0, 47.3427..."
2,tile00003,"[[-124.5, 40.125843708678985], [-124.5, 39.548..."
3,tile00004,"[[-124.5, 41.85789451624787], [-124.5, 41.2805..."
4,tile00005,"[[-124.5, 43.58994532381674], [-124.5, 43.0125..."


In [7]:
print(f"Rows ingested: {len(df):,}")
print("Columns:", list(df.columns))
print("First polygon vertex count:", len(df.loc[0, "PolygonCoordinates"]))

Rows ingested: 1,043
Columns: ['Taxonomy', 'Tag', 'CenterLon', 'CenterLat', 'Coordinates', 'PolygonCoordinates']
First polygon vertex count: 6


In [8]:
aTagToCoords = {row["Tag"]: row["PolygonCoordinates"] for _, row in df.iterrows()}
list(aTagToCoords.items())[100:102]

[('tile00101',
  [[-118.5, 45.32199613138562],
   [-118.5, 44.744645862195995],
   [-118.0, 44.45597072760118],
   [-117.5, 44.744645862195995],
   [-117.5, 45.32199613138562],
   [-118.0, 45.61067126598043]]),
 ('tile00102',
  [[-118.5, 47.05404693895449],
   [-118.5, 46.47669666976487],
   [-118.0, 46.18802153517006],
   [-117.5, 46.47669666976487],
   [-117.5, 47.05404693895449],
   [-118.0, 47.342722073549304]])]

---

## Plotting

In [9]:
def plot_polygons(df, bg_color="black"):
    fig = go.Figure()

    for poly in df["PolygonCoordinates"]:
        xs = [p[0] for p in poly] + [poly[0][0]]
        ys = [p[1] for p in poly] + [poly[0][1]]

        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                fill="toself",
                line=dict(color="white"),
                fillcolor="rgba(188,188,188,0.3)",
                hoverinfo="skip",
                showlegend=False
            )
        )

    fig.update_layout(
        width=1000,
        height=1000,
        paper_bgcolor=bg_color,  # outside plot area
        plot_bgcolor=bg_color,   # inside plot area
        xaxis=dict(scaleanchor="y"),
    )

    # optional: hide axes/ticks
    fig.update_xaxes(visible=True)
    fig.update_yaxes(visible=True)

    return fig

# examples
#plot_polygons(df, bg_color="silver").show()
plot_polygons(df, bg_color="#1F1F1F").show()